<a href="https://colab.research.google.com/github/Luwii22/HTML/blob/main/CYB304_Tasks_1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CYB 304 Lab: Tasks 1-3 (Spark pipeline)
Dataset: IT_customer_churn.csv (telecom churn). Run cells top to bottom and **screenshot each output** for your report.

In [13]:
import glob, os, shutil
found = glob.glob("/content/IT_customer_churn*.csv") + glob.glob("/IT_customer_churn*.csv")
print("Found:", found)
if not os.path.exists("/content/IT_customer_churn.csv"):
    shutil.copy(found[0], "/content/IT_customer_churn.csv")
raw_file = "/content/IT_customer_churn.csv"
print("Ready:", os.path.exists(raw_file))

Found: ['/content/IT_customer_churn.csv.csv']
Ready: True


## 0. Start Spark

In [10]:
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.appName("CYB304_BigData_Lab")
         .master("local[*]").config("spark.sql.shuffle.partitions", "8").getOrCreate())
print("Spark version:", spark.version)
print("Cores available:", spark.sparkContext.defaultParallelism)

Spark version: 4.0.5
Cores available: 2


**1b. Inspect the raw file** (structure, size, quality) before ingesting.

**1c. Create several data sources and load them into a distributed storage layer.**
Colab has no HDFS, so `/content/datalake` is our storage zone. Spark writes it in the same partitioned layout HDFS uses. In your report, state that on a real cluster the path would be `hdfs://namenode:9000/datalake/...`.
We split the data into two sources of different formats (a CSV customer profile and a JSON billing feed) to simulate multiple sources.

In [14]:
from pyspark.sql.window import Window

base = spark.read.csv(raw_file, header=True, inferSchema=True)
w = Window.orderBy(F.monotonically_increasing_id())
base = base.withColumn("customer_id", F.row_number().over(w)).cache()
print("Rows ingested:", base.count())

LAKE = "/content/datalake"
profile_cols = ["customer_id","gender","SeniorCitizen","Partner","Dependents","tenure","Churn"]
billing_cols = ["customer_id","Contract","PaperlessBilling","PaymentMethod","MonthlyCharges","TotalCharges",
                "PhoneService","MultipleLines","InternetService","OnlineSecurity","OnlineBackup",
                "DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

# Source 1: CSV (customer profile). Source 2: JSON (billing & services)
base.select(*profile_cols).write.mode("overwrite").option("header", True).csv(f"{LAKE}/raw/profile_csv")
base.select(*billing_cols).write.mode("overwrite").json(f"{LAKE}/raw/billing_json")
print("Written to the storage zone.")

Rows ingested: 7043
Written to the storage zone.


**1d. Verify ingestion by listing and sampling the stored files.**

In [15]:
import subprocess
print(subprocess.run(["ls","-lhR",f"{LAKE}/raw"],capture_output=True,text=True).stdout)

/content/datalake/raw:
total 8.0K
drwxr-xr-x 2 root root 4.0K Nov 13 21:21 billing_json
drwxr-xr-x 2 root root 4.0K Nov 13 21:21 profile_csv

/content/datalake/raw/billing_json:
total 2.6M
-rw-r--r-- 1 root root 2.6M Nov 13 21:21 part-00000-9f7df882-6523-442b-8447-887da93eac01-c000.json
-rw-r--r-- 1 root root    0 Nov 13 21:21 _SUCCESS

/content/datalake/raw/profile_csv:
total 180K
-rw-r--r-- 1 root root 177K Nov 13 21:21 part-00000-7000eb33-ffe6-4dea-a137-d0225f39e8d5-c000.csv
-rw-r--r-- 1 root root    0 Nov 13 21:21 _SUCCESS



In [16]:
print("Profile (CSV) sample:")
spark.read.csv(f"{LAKE}/raw/profile_csv", header=True, inferSchema=True).show(5)
print("Billing (JSON) sample:")
spark.read.json(f"{LAKE}/raw/billing_json").show(5, truncate=False)

Profile (CSV) sample:
+-----------+------+-------------+-------+----------+------+-----+
|customer_id|gender|SeniorCitizen|Partner|Dependents|tenure|Churn|
+-----------+------+-------------+-------+----------+------+-----+
|          1|Female|            0|    Yes|        No|     1|   No|
|          2|  Male|            0|     No|        No|    34|   No|
|          3|  Male|            0|     No|        No|     2|  Yes|
|          4|  Male|            0|     No|        No|    45|   No|
|          5|Female|            0|     No|        No|     2|  Yes|
+-----------+------+-------------+-------+----------+------+-----+
only showing top 5 rows
Billing (JSON) sample:
+--------------+----------------+---------------+--------------+----------------+------------+--------------+----------------+-------------------------+------------+---------------+-----------+-----------+------------+-----------+
|Contract      |DeviceProtection|InternetService|MonthlyCharges|MultipleLines   |OnlineBackup|Onl

## Task 2: Data discovery on raw data
**2a. Read directly from storage and combine the two sources.**

In [17]:
profile = spark.read.csv(f"{LAKE}/raw/profile_csv", header=True, inferSchema=True)
billing = spark.read.json(f"{LAKE}/raw/billing_json")
df = profile.join(billing, "customer_id", "inner")
print("Rows:", df.count(), "| Columns:", len(df.columns))
df.printSchema()

Rows: 7043 | Columns: 21
root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- Churn: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- TotalCharges: string (nullable = true)



**2b. Summary statistics and sampling.**

In [18]:
df.select("tenure","MonthlyCharges").describe().show()
df.sample(fraction=0.01, seed=42).show(5)

+-------+------------------+------------------+
|summary|            tenure|    MonthlyCharges|
+-------+------------------+------------------+
|  count|              7043|              7043|
|   mean| 32.37114865824223| 64.76169246059922|
| stddev|24.559481023094442|30.090047097678482|
|    min|                 0|             18.25|
|    max|                72|            118.75|
+-------+------------------+------------------+

+-----------+------+-------------+-------+----------+------+-----+--------------+-------------------+---------------+--------------+----------------+-------------------+-------------------+----------------+--------------------+------------+-------------------+-------------------+-------------------+------------+
|customer_id|gender|SeniorCitizen|Partner|Dependents|tenure|Churn|      Contract|   DeviceProtection|InternetService|MonthlyCharges|   MultipleLines|       OnlineBackup|     OnlineSecurity|PaperlessBilling|       PaymentMethod|PhoneService|    Streaming

**2c. Data quality checks.**
Note: `TotalCharges` may load as text because some values are blank. We use `try_cast` because Spark 4 runs in ANSI mode.

In [19]:
# Null counts per column
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

# Blank strings hidden in text columns
blank_total = df.filter(F.trim(F.col("TotalCharges").cast("string")) == "").count()
print("Blank TotalCharges values:", blank_total)

# Who are they?
df.filter(F.trim(F.col("TotalCharges").cast("string")) == "") \
  .select("customer_id","tenure","MonthlyCharges","TotalCharges","Contract").show()

# Duplicates (ignoring the id we created)
cols_wo_id = [c for c in df.columns if c != "customer_id"]
print("Total rows:", df.count(), "| Distinct rows:", df.dropDuplicates(cols_wo_id).count())

-RECORD 0---------------
 customer_id      | 0   
 gender           | 0   
 SeniorCitizen    | 0   
 Partner          | 0   
 Dependents       | 0   
 tenure           | 0   
 Churn            | 0   
 Contract         | 0   
 DeviceProtection | 0   
 InternetService  | 0   
 MonthlyCharges   | 0   
 MultipleLines    | 0   
 OnlineBackup     | 0   
 OnlineSecurity   | 0   
 PaperlessBilling | 0   
 PaymentMethod    | 0   
 PhoneService     | 0   
 StreamingMovies  | 0   
 StreamingTV      | 0   
 TechSupport      | 0   
 TotalCharges     | 0   

Blank TotalCharges values: 11
+-----------+------+--------------+------------+--------+
|customer_id|tenure|MonthlyCharges|TotalCharges|Contract|
+-----------+------+--------------+------------+--------+
|        489|     0|         52.55|            |Two year|
|        754|     0|         20.25|            |Two year|
|        937|     0|         80.85|            |Two year|
|       1083|     0|         25.75|            |Two year|
|       1341|

**2d. Patterns and class balance.**

In [20]:
df.groupBy("Churn").count().withColumn("pct", F.round(F.col("count")/df.count()*100,1)).show()
df.groupBy("Contract").agg(
    F.count("*").alias("customers"),
    F.round(F.avg(F.when(F.col("Churn")=="Yes",1).otherwise(0))*100,1).alias("churn_rate_pct")
).orderBy("churn_rate_pct", ascending=False).show()

+-----+-----+----+
|Churn|count| pct|
+-----+-----+----+
|   No| 5174|73.5|
|  Yes| 1869|26.5|
+-----+-----+----+

+--------------+---------+--------------+
|      Contract|customers|churn_rate_pct|
+--------------+---------+--------------+
|Month-to-month|     3875|          42.7|
|      One year|     1473|          11.3|
|      Two year|     1695|           2.8|
+--------------+---------+--------------+



**Write your Task 2 observations here (for the report):**
- TotalCharges has blank values that stop it being numeric; they need cleaning.
- Duplicate rows exist and should be removed.
- Churn is imbalanced (about 27% Yes).
- Month-to-month contracts churn far more than long contracts.

## Task 3: Big Data analytics with Spark
**3a. Clean the data (transformations).**

In [21]:
clean = (df.dropDuplicates(cols_wo_id)
           .withColumn("TotalCharges", F.expr("try_cast(trim(TotalCharges) as double)"))
           .withColumn("TotalCharges", F.coalesce(F.col("TotalCharges"), F.lit(0.0)))
           .withColumn("tenure_group",
                F.when(F.col("tenure") <= 12, "0-12 months")
                 .when(F.col("tenure") <= 24, "13-24 months")
                 .when(F.col("tenure") <= 48, "25-48 months")
                 .otherwise("49+ months"))
           .cache())
print("Clean rows:", clean.count())
clean.printSchema()

Clean rows: 7021
root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- Churn: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- TotalCharges: double (nullable = false)
 |-- tenure_group: string (nullable = false)



**3b. Filtering, aggregation and grouping.**

In [22]:
churn_flag = F.when(F.col("Churn")=="Yes",1).otherwise(0)

print("Filter: high-spend customers (MonthlyCharges > 90) who churned")
print(clean.filter((F.col("MonthlyCharges")>90) & (F.col("Churn")=="Yes")).count())

print("Churn by tenure group")
clean.groupBy("tenure_group").agg(F.count("*").alias("customers"),
    F.round(F.avg(churn_flag)*100,1).alias("churn_rate_pct")).orderBy("tenure_group").show()

print("Churn and revenue by internet service")
clean.groupBy("InternetService").agg(F.count("*").alias("customers"),
    F.round(F.avg("MonthlyCharges"),2).alias("avg_monthly"),
    F.round(F.avg(churn_flag)*100,1).alias("churn_rate_pct")).show()

print("Churn by payment method")
clean.groupBy("PaymentMethod").agg(F.count("*").alias("customers"),
    F.round(F.avg(churn_flag)*100,1).alias("churn_rate_pct")).orderBy("churn_rate_pct", ascending=False).show(truncate=False)

Filter: high-spend customers (MonthlyCharges > 90) who churned
570
Churn by tenure group
+------------+---------+--------------+
|tenure_group|customers|churn_rate_pct|
+------------+---------+--------------+
| 0-12 months|     2164|          47.4|
|13-24 months|     1024|          28.7|
|25-48 months|     1594|          20.4|
|  49+ months|     2239|           9.5|
+------------+---------+--------------+

Churn and revenue by internet service
+---------------+---------+-----------+--------------+
|InternetService|customers|avg_monthly|churn_rate_pct|
+---------------+---------+-----------+--------------+
|             No|     1512|      21.09|           7.2|
|    Fiber optic|     3090|      91.54|          41.8|
|            DSL|     2419|      58.11|          18.9|
+---------------+---------+-----------+--------------+

Churn by payment method
+-------------------------+---------+--------------+
|PaymentMethod            |customers|churn_rate_pct|
+-------------------------+---------

**3c. Spark SQL (same engine, SQL syntax).**

In [23]:
clean.createOrReplaceTempView("customers")
spark.sql('''
    SELECT Contract, InternetService, COUNT(*) AS customers,
           ROUND(AVG(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END)*100,1) AS churn_pct
    FROM customers GROUP BY Contract, InternetService ORDER BY churn_pct DESC
''').show(truncate=False)

+--------------+---------------+---------+---------+
|Contract      |InternetService|customers|churn_pct|
+--------------+---------------+---------+---------+
|Month-to-month|Fiber optic    |2122     |54.5     |
|Month-to-month|DSL            |1221     |32.1     |
|One year      |Fiber optic    |539      |19.3     |
|Month-to-month|No             |510      |18.6     |
|One year      |DSL            |570      |9.3      |
|Two year      |Fiber optic    |429      |7.2      |
|One year      |No             |364      |2.5      |
|Two year      |DSL            |628      |1.9      |
|Two year      |No             |638      |0.8      |
+--------------+---------------+---------+---------+



**3d. Batch (MapReduce-style) vs in-memory (Spark).**
First a classic map/reduce count using the RDD API, which mirrors Hadoop MapReduce's map -> shuffle -> reduce. Then the same job timed with and without caching.

In [24]:
# MapReduce-style: map each row to (Contract, 1), then reduce by key
counts = (clean.rdd.map(lambda r: (r["Contract"], 1))
                   .reduceByKey(lambda a, b: a + b)
                   .collect())
print(counts)

[('Month-to-month', 3853), ('One year', 1473), ('Two year', 1695)]


In [25]:
import time
def job(frame):
    t = time.time()
    frame.groupBy("Contract").agg(F.avg("MonthlyCharges")).collect()
    frame.groupBy("PaymentMethod").agg(F.avg("tenure")).collect()
    frame.groupBy("InternetService").agg(F.sum("TotalCharges")).collect()
    return time.time() - t

from_disk = spark.read.csv(raw_file, header=True, inferSchema=True)   # re-reads from storage each action
t_disk = job(from_disk)

mem = clean.cache(); mem.count()                                       # data held in memory
t_mem = job(mem)

print(f"Re-reading from disk each time: {t_disk:.2f}s")
print(f"In-memory (cached):             {t_mem:.2f}s")

{"ts": "2026-11-13 21:25:47.200", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' ' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor68.invoke(Unknown Source)", "line": "", "fragment": "sum", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o587.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' ' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"sum\" was called from\njdk.internal.reflect.Ge

NumberFormatException: [CAST_INVALID_INPUT] The value ' ' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"sum" was called from
jdk.internal.reflect.GeneratedMethodAccessor68.invoke(Unknown Source)


**Write your Task 3 discussion here:**
- Hadoop MapReduce writes intermediate results to disk between stages; Spark keeps them in memory (RDD/DataFrame caching), so iterative and interactive work is much faster.
- Spark builds a lazy DAG of transformations and only runs on actions, so it can optimise the plan (Catalyst).
- Both scale horizontally by adding nodes; Hadoop is cheaper for very large, infrequent batch jobs, Spark for iterative analytics and ML.
- With only about 7k rows the timing difference is small; the principle holds at scale.

## Export the cleaned data for Task 5 (R) and Task 6 (MongoDB)
This writes one CSV and downloads it, then use it in RStudio.

In [26]:
final = clean.drop("tenure_group")
final.toPandas().to_csv("/content/churn_clean.csv", index=False)
files.download("/content/churn_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
spark.stop()